# STEP 1 — Install Required Libraries

In [1]:
!pip -q install kagglehub
!pip -q install segmentation-models-pytorch
!pip -q install albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.6 MB/s eta 0:00:00


# STEP 2 — Imports and Reproducibility Setup

In [2]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

from tqdm.auto import tqdm

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: Tesla T4


# STEP 3 — Record Environment Versions

In [4]:
import albumentations
import segmentation_models_pytorch

print("PyTorch:", torch.__version__)
print(
    "Albumentations:",
    albumentations.__version__
)
print(
    "Segmentation Models PyTorch:",
    segmentation_models_pytorch.__version__
)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

PyTorch: 2.11.0+cu128
Albumentations: 2.0.8
Segmentation Models PyTorch: 0.5.0
OpenCV: 5.0.0
NumPy: 2.1.3
Pandas: 2.2.3


# STEP 4 — Download PlantVillage Dataset

In [5]:
import kagglehub

download_path = kagglehub.dataset_download(
    "abdallahalidev/plantvillage-dataset"
)

print(
    "Downloaded dataset path:"
)

print(download_path)

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Downloaded dataset path:
/kaggle/input/plantvillage-dataset


# STEP 5 — Detect Dataset Root

In [6]:
base_path = Path(download_path)


def find_dataset_root(base):

    candidates = [base]

    candidates.extend(
        p
        for p in base.rglob("*")
        if p.is_dir()
    )

    for path in candidates:

        try:

            folders = {
                x.name
                for x in path.iterdir()
                if x.is_dir()
            }

            required = {
                "color",
                "grayscale",
                "segmented"
            }

            if required.issubset(
                folders
            ):
                return path

        except PermissionError:
            continue

    return None


dataset_root = find_dataset_root(
    base_path
)

print(
    "Dataset root:",
    dataset_root
)

Dataset root: /kaggle/input/plantvillage-dataset/plantvillage dataset


In [7]:
assert dataset_root is not None

assert (
    dataset_root / "color"
).exists()

assert (
    dataset_root / "segmented"
).exists()

print(
    "✅ Dataset structure found."
)

✅ Dataset structure found.


# STEP 6 — Mount Google Drive

In [8]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


In [9]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "PlantVillage_Segmentation"
)

print(PROJECT_DIR)

assert PROJECT_DIR.exists()

/content/drive/MyDrive/PlantVillage_Segmentation


# STEP 7 — Load Final Preprocessing Artifacts

In [10]:
FINAL_MANIFEST = (
    PROJECT_DIR /
    "final_segmentation_manifest.csv"
)

FINAL_CONFIG = (
    PROJECT_DIR /
    "segmentation_preprocessing_config.json"
)


print(
    "Manifest exists:",
    FINAL_MANIFEST.exists()
)

print(
    "Config exists:",
    FINAL_CONFIG.exists()
)

Manifest exists: True
Config exists: True


In [11]:
final_df = pd.read_csv(
    FINAL_MANIFEST
)


with open(
    FINAL_CONFIG,
    "r"
) as f:

    preprocessing_config = (
        json.load(f)
    )


print(
    "Total samples:",
    len(final_df)
)

print()

print(
    final_df[
        "split"
    ].value_counts()
)

print()

print(
    "Classes:",
    final_df[
        "class"
    ].nunique()
)

Total samples: 52969

split
train    42371
val       5300
test      5298
Name: count, dtype: int64

Classes: 38


# STEP 8 — Verify Final Preprocessing Configuration

In [12]:
BLACK_THRESHOLD = (
    preprocessing_config[
        "black_threshold"
    ]
)

IMAGE_SIZE = (
    preprocessing_config[
        "input_image_size"
    ][0]
)


print(
    "Black threshold:",
    BLACK_THRESHOLD
)

print(
    "Image size:",
    IMAGE_SIZE
)

Black threshold: 15
Image size: 256


# STEP 9 — Verify Manifest Paths

In [13]:
sample_check = final_df.sample(
    n=10,
    random_state=42
)


missing_color = 0
missing_segmented = 0


for _, row in sample_check.iterrows():

    color_path = (
        dataset_root /
        row["color_rel"]
    )

    segmented_path = (
        dataset_root /
        row["segmented_rel"]
    )


    if not color_path.exists():
        missing_color += 1


    if not segmented_path.exists():
        missing_segmented += 1


print(
    "Missing color files:",
    missing_color
)

print(
    "Missing segmented files:",
    missing_segmented
)

Missing color files: 0
Missing segmented files: 0


# STEP 10 — Final Segmentation Target Function

In [14]:
def create_final_segmentation_target(
    segmented_rgb,
    target_height,
    target_width,
    black_threshold=15
):

    """
    Create binary segmentation target
    from a QC-approved PlantVillage
    segmented RGB image.

    Background = 0
    Leaf       = 1

    No binary mask is saved to disk.
    """

    segmented_rgb = np.asarray(
        segmented_rgb,
        dtype=np.uint8
    )


    # Foreground extraction
    mask = (
        np.max(
            segmented_rgb,
            axis=2
        )
        >
        black_threshold
    ).astype(np.uint8)


    # Resize binary labels only
    # if spatial alignment is required
    if mask.shape != (
        target_height,
        target_width
    ):

        mask = cv2.resize(
            mask,
            (
                target_width,
                target_height
            ),
            interpolation=cv2.INTER_NEAREST
        )


    return mask

# STEP 11 — Define Common Data Transformations

In [15]:
train_transform = A.Compose([

    A.Resize(
        IMAGE_SIZE,
        IMAGE_SIZE
    ),

    A.HorizontalFlip(
        p=0.5
    ),

    A.VerticalFlip(
        p=0.5
    ),

    A.RandomRotate90(
        p=0.5
    ),

    A.Normalize(
        mean=(
            0.485,
            0.456,
            0.406
        ),
        std=(
            0.229,
            0.224,
            0.225
        )
    ),

    ToTensorV2()
])


eval_transform = A.Compose([

    A.Resize(
        IMAGE_SIZE,
        IMAGE_SIZE
    ),

    A.Normalize(
        mean=(
            0.485,
            0.456,
            0.406
        ),
        std=(
            0.229,
            0.224,
            0.225
        )
    ),

    ToTensorV2()
])

# STEP 12 — Create Segmentation Dataset Class

In [16]:
class PlantVillageSegmentationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        dataset_root,
        transform=None,
        black_threshold=15
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.dataset_root = Path(
            dataset_root
        )

        self.transform = transform

        self.black_threshold = (
            black_threshold
        )


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]


        color_path = (
            self.dataset_root /
            row["color_rel"]
        )


        segmented_path = (
            self.dataset_root /
            row["segmented_rel"]
        )


        # ---------------------------------
        # Model input
        # ---------------------------------

        image = np.array(
            Image.open(
                color_path
            ).convert("RGB")
        )


        # ---------------------------------
        # PlantVillage supervision source
        # ---------------------------------

        segmented = np.array(
            Image.open(
                segmented_path
            ).convert("RGB")
        )


        # ---------------------------------
        # Dynamic segmentation target
        # ---------------------------------

        mask = (
            create_final_segmentation_target(
                segmented_rgb=segmented,
                target_height=image.shape[0],
                target_width=image.shape[1],
                black_threshold=self.black_threshold
            )
        )


        # ---------------------------------
        # Joint transformations
        # ---------------------------------

        if self.transform is not None:

            transformed = (
                self.transform(
                    image=image,
                    mask=mask
                )
            )

            image = transformed[
                "image"
            ]

            mask = transformed[
                "mask"
            ]


        mask = mask.float()


        if mask.ndim == 2:

            mask = mask.unsqueeze(0)


        return {
            "image": image,

            "mask": mask,

            "class_name":
                row["class"],

            "color_rel":
                row["color_rel"],

            "segmented_rel":
                row["segmented_rel"]
        }

# STEP 13 — Create Final Dataset Splits

In [17]:
train_df = final_df[
    final_df["split"]
    ==
    "train"
].copy()


val_df = final_df[
    final_df["split"]
    ==
    "val"
].copy()


test_df = final_df[
    final_df["split"]
    ==
    "test"
].copy()


print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

print(
    "Test:",
    len(test_df)
)

Train: 42371
Validation: 5300
Test: 5298


In [18]:
train_dataset = (
    PlantVillageSegmentationDataset(
        dataframe=train_df,
        dataset_root=dataset_root,
        transform=train_transform,
        black_threshold=BLACK_THRESHOLD
    )
)


val_dataset = (
    PlantVillageSegmentationDataset(
        dataframe=val_df,
        dataset_root=dataset_root,
        transform=eval_transform,
        black_threshold=BLACK_THRESHOLD
    )
)


test_dataset = (
    PlantVillageSegmentationDataset(
        dataframe=test_df,
        dataset_root=dataset_root,
        transform=eval_transform,
        black_threshold=BLACK_THRESHOLD
    )
)

# STEP 14 — Create GPU DataLoaders

In [19]:
BATCH_SIZE = 16
NUM_WORKERS = 2


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=True
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=True
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=True
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

Train batches: 2649
Validation batches: 332
Test batches: 332


# STEP 15 — DataLoader Sanity Check

In [20]:
batch = next(
    iter(train_loader)
)


images = batch[
    "image"
]

masks = batch[
    "mask"
]


print(
    "Images:",
    images.shape
)

print(
    "Masks:",
    masks.shape
)

print(
    "Image dtype:",
    images.dtype
)

print(
    "Mask dtype:",
    masks.dtype
)

print(
    "Mask values:",
    torch.unique(masks)
)

Images: torch.Size([16, 3, 256, 256])
Masks: torch.Size([16, 1, 256, 256])
Image dtype: torch.float32
Mask dtype: torch.float32
Mask values: tensor([0., 1.])


# STEP 16 — GPU Batch Test

In [21]:
images_gpu = images.to(
    DEVICE,
    non_blocking=True
)

masks_gpu = masks.to(
    DEVICE,
    non_blocking=True
)


print(
    "Image device:",
    images_gpu.device
)

print(
    "Mask device:",
    masks_gpu.device
)

print(
    "GPU memory allocated (MB):",
    torch.cuda.memory_allocated()
    / 1024**2
)

Image device: cuda:0
Mask device: cuda:0
GPU memory allocated (MB): 16.0
